# Real Data Pipeline - Analytics/Gold Layer
## Business-ready analytics and insights


In [0]:
# Purpose: Create business-ready analytics tables
 
import pyspark.sql.functions as F
from pyspark.sql.window import Window
 
catalog = "workspace"
schema = "github_analytics"
 
print("=" * 70)
print("GITHUB PIPELINE - ANALYTICS LAYER")
print("=" * 70)

In [0]:
# Cell 1: Load Silver Tables
silver_repos = spark.table(f"{catalog}.{schema}.silver_repositories")
silver_contribs = spark.table(f"{catalog}.{schema}.silver_contributors")
 
print("✅ Loaded silver tables")
 

In [0]:
# Cell 2: Gold Table 1 - Repository Rankings
print("\n" + "=" * 70)
print("CREATING GOLD TABLES")
print("=" * 70)

gold_repo_rankings = silver_repos \
    .select(
        "repo_name", "owner", "repo", "stars", "forks", "watchers",
        "language", "is_active", "days_since_update", "popularity_score"
    ) \
    .withColumn(
        "overall_rank",
        F.row_number().over(Window.orderBy(F.col("stars").desc()))
    ) \
    .withColumn(
        "language_rank",
        F.row_number().over(
            Window.partitionBy("language").orderBy(F.col("stars").desc())
        )
    ) \
    .withColumn(
        "tier",
        F.when(F.col("stars") >= 10000, "platinum")
         .when(F.col("stars") >= 5000, "gold")
         .when(F.col("stars") >= 1000, "silver")
         .otherwise("bronze")
    )

gold_table1 = f"{catalog}.{schema}.gold_repository_rankings"
gold_repo_rankings.write.format("delta").mode("overwrite").saveAsTable(gold_table1)

print(f"✅ Created: {gold_table1} ({gold_repo_rankings.count()} records)")


In [0]:
# Cell 3: Gold Table 2 - Contributor Analysis
gold_contributor_analysis = silver_repos.alias("r") \
    .join(
        silver_contribs.alias("c"),
        F.col("r.repo_name") == F.col("c.repo_name"),
        "inner"
    ) \
    .groupBy("c.contributor_login") \
    .agg(
        F.count("r.repo_name").alias("repos_contributed"),
        F.sum("c.contributions").alias("total_contributions"),
        F.collect_list("r.repo_name").alias("repos_list"),
        F.avg("r.stars").alias("avg_repo_stars"),
        F.max("r.stars").alias("max_repo_stars")
    ) \
    .withColumn(
        "expertise_level",
        F.when(F.col("total_contributions") >= 500, "expert")
         .when(F.col("total_contributions") >= 100, "experienced")
         .otherwise("beginner")
    ) \
    .withColumn(
        "contributor_rank",
        F.row_number().over(Window.orderBy(F.col("total_contributions").desc()))
    )

gold_table2 = f"{catalog}.{schema}.gold_contributor_analysis"
gold_contributor_analysis.write.format("delta").mode("overwrite").saveAsTable(gold_table2)

print(f"✅ Created: {gold_table2} ({gold_contributor_analysis.count()} records)")
